# Yağış Veri Seti ile İkili Tahmin

Meteorolojik verileri kullanarak o gün yağmur yağıp yağmayacağını (0 veya 1) tahmin etmeye dayalı bir proğram yapacağız.  Amacımız, hava durumu ölçümlerine bakarak yağmur yağıp yağmayacağını (0: Yağmur Yok, 1: Yağmurlu) tahmin etmektir.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
train= pd.read_csv('train(5).csv')
test= pd.read_csv('test(5).csv')

In [3]:
train.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


In [4]:
test.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
0,2190,1,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,24.3
1,2191,2,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,35.3
2,2192,3,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,16.9
3,2193,4,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,50.6
4,2194,5,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,19.4


In [5]:
HEDEF = 'rainfall'

In [6]:
# 2. Meteorolojik Özellik Mühendisliği (Hava Durumu Formülleri)
for df in [train, test]:
    # Sıcaklık Farkı: En yüksek ve en düşük sıcaklık arasındaki makas
    df['temp_range'] = df['maxtemp'] - df['mintemp']
    
    # Çiy Noktası Açığı: Sıcaklık çiy noktasına ne kadar yakınsa yağmur ihtimali o kadar artar
    df['dewpoint_depression'] = df['temparature'] - df['dewpoint']
    
    # Gökyüzü Kapanma Oranı: Bulutluluk oranı ile güneşlenme süresi etkileşimi
    df['sky_coverage_index'] = df['cloud'] / (df['sunshine'] + 1e-5)
    
    # Rüzgar ve Nem Etkileşimi
    df['wind_humidity_interaction'] = df['windspeed'] * df['humidity']

print("2. Adım: Hava durumu özellik mühendisliği tamamlandı. Modelleme başlatılıyor...")


2. Adım: Hava durumu özellik mühendisliği tamamlandı. Modelleme başlatılıyor...


In [7]:
# Girdileri (X) ve Hedefi (y) Kesin Olarak Ayırma
# Hedef kolonumuz 'rainfall'dur. Giriş sütunlarında bu ve 'id' olmayacak.
giris_sutunlari = [kolon for kolon in train.columns if kolon not in ['id', HEDEF]]

X = train[giris_sutunlari]
y = train[HEDEF]
X_test = test[giris_sutunlari] # Train ile tamamen aynı sütunlar seçildi, hata vermez!


In [8]:
# 5-Fold Stratified Cross Validation Düzeni
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_tahminleri = np.zeros(len(train))
test_tahminleri = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Sınıflandırma için LightGBM kurulumu
    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.04,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    
    # ROC-AUC için 1 sınıfına (Yağmur Var) ait olasılıkları tahmin ediyoruz
    oof_tahminleri[val_idx] = model.predict_proba(X_val)[:, 1]
    test_tahminleri += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Katman skoru hesaplama
    fold_auc = roc_auc_score(y_val, oof_tahminleri[val_idx])
    print(f"Katman {fold + 1} ROC-AUC Skoru: {fold_auc:.4f}")

# Genel Başarı Oranı
genel_auc = roc_auc_score(y, oof_tahminleri)
print(f"\n---> TÜM VERİ SETİ GENEL ROC-AUC SKORU: {genel_auc:.4f} <---")


Katman 1 ROC-AUC Skoru: 0.9070
Katman 2 ROC-AUC Skoru: 0.8322
Katman 3 ROC-AUC Skoru: 0.8419
Katman 4 ROC-AUC Skoru: 0.8820
Katman 5 ROC-AUC Skoru: 0.8718

---> TÜM VERİ SETİ GENEL ROC-AUC SKORU: 0.8654 <---


In [9]:
# Kaggle Gönderi Dosyasının Oluşturulması
submission = pd.DataFrame({
    'id': test['id'],
    'rainfall': test_tahminleri
})


In [10]:
submission.to_csv('submission_rainfall.csv', index=False)